In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt


In [ ]:
data_set = datasets.MNIST(root='data',
                      train=True,
                      download=True,
                      transform=v2.Compose([
                          v2.ToTensor(),
                          v2.Normalize((0.1307,), (0.3081,),
                          )
                      ]))
data_set1 = datasets.MNIST(root='data',
                      train=False,
                      download=True,
                      transform=v2.Compose([
                          v2.ToTensor(),
                          v2.Normalize((0.1307,), (0.3081,)),

                      ]))

In [ ]:
ldr = DataLoader(data_set, shuffle = True, batch_size = 64)
tstlsr = DataLoader(data_set1, shuffle = True, batch_size = 64)

In [ ]:
def vis(ldr, n = 10):
  images, labels = next(iter(ldr))
  fig = plt.figure(figsize = (10, 10))
  for i in range(n):
    plt.subplot(int(n/3) +1,3,i+1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")

  plt.tight_layout()
  plt.show


In [ ]:
vis(ldr)

In [ ]:
import torch
import torch.nn as nn

class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
device = "cuda"
model = MNISTCNN().to(device)

lf = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.8, patience=2)

In [ ]:
def train(ei):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, (img, lbl) in enumerate(ldr):
        img, lbl = img.to(device), lbl.to(device)

        optimizer.zero_grad()

        outputs = model(img)

        loss = lf(outputs, lbl)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += lbl.size(0)
        correct += (predicted == lbl).sum().item()

    epoch_loss = running_loss / len(ldr)
    epoch_acc = 100.0 * correct / total
    print(f"\n Epoch {ei} Loss = {epoch_loss:.4f} | Accuracy = {epoch_acc:.2f}%")
    return epoch_loss

In [ ]:
def evaluate_model():
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for img, lbl in tstlsr:
            img, lbl = img.to(device), lbl.to(device)

            outputs = model(img)
            loss = lf(outputs, lbl)

            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += lbl.size(0)
            correct += (predicted == lbl).sum().item()

    avg_test_loss = test_loss / len(tstlsr)
    test_accuracy = 100.0 * correct / total
    print(f"Test Loss = {avg_test_loss:.4f} | Accuracy = {test_accuracy:.2f}%\n")

    return test_accuracy

In [ ]:
import matplotlib.pyplot as plt

training_losses = []
test_accuracies = []

for epoch in range(1, 25):

    current_train_loss = train(epoch)
    val_accuracy = evaluate_model()

    training_losses.append(current_train_loss)
    test_accuracies.append(val_accuracy)

    scheduler.step(val_accuracy)


    plt.figure(figsize=(10, 5))
    plt.plot(range(1, epoch + 1), training_losses, label='Training Loss')
    plt.plot(range(1, epoch + 1), test_accuracies, label='Test Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.title('Training Loss and Test Accuracy per Epoch')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
torch.save(model.state_dict(), 'model2_24epochs.pth')

In [ ]:
import random

inferenceModel = MNISTCNN()
inferenceModel.load_state_dict(torch.load('model2_24epochs.pth'))
inferenceModel.to(device)
inferenceModel.eval()

smpl, smpl_lbl = data_set1[random.randint(0,len(data_set1)-1)]
input = smpl.unsqueeze(0).to(device)
with torch.no_grad():
    raw_logits = inferenceModel(input)
    prediction = torch.argmax(raw_logits, dim=1).item()

display_img = smpl.squeeze().cpu()
plt.imshow(display_img, cmap='gray')
plt.title(f"Model Prediction: {prediction} | True Label: {smpl_lbl}")
plt.axis('off')
plt.show()